In [1]:
from pathlib import Path
import partitura as pt

musicxml_path = Path("dataset/raw/beethoven-moonlight-sonata-1st-mvt.mxl")

score = pt.load_score(str(musicxml_path))

try:
    note_array = score.note_array(include_staff=True)
except TypeError:
    note_array = score.note_array()

print("Available fields:")
print(note_array.dtype.names)

print("\nFirst 20 notes:")
for i, n in enumerate(note_array[:20]):
    row = {}

    for field in note_array.dtype.names:
        value = n[field]
        if hasattr(value, "item"):
            value = value.item()
        row[field] = value

    print(i, row)

c:\Users\User\wat_aibuilder\.venv\Lib\site-packages\partitura\directions.py:524: UserWarning: error parsing "Si deve suonare tutto questo pezzo delicatissimamente e senza sordini" (UnexpectedCharacters)
  warnings.warn('error parsing "{}" ({})'.format(string, type(e).__name__))
c:\Users\User\wat_aibuilder\.venv\Lib\site-packages\partitura\directions.py:524: UserWarning: error parsing "sempre        e senza sordini" (UnexpectedCharacters)
  warnings.warn('error parsing "{}" ({})'.format(string, type(e).__name__))
c:\Users\User\wat_aibuilder\.venv\Lib\site-packages\partitura\io\importmusicxml.py:999: UserWarning: Did not find a wedge start element for wedge stop!
  warnings.warn("Did not find a wedge start element for wedge stop!")
c:\Users\User\wat_aibuilder\.venv\Lib\site-packages\partitura\io\importmusicxml.py:1077: UserWarning: ignoring direction type: metronome {'parentheses': 'no', 'relative-y': '20.00'}
  warnings.warn("ignoring direction type: {} {}".format(dt.tag, dt.attrib))
c:

Available fields:
('onset_beat', 'duration_beat', 'onset_quarter', 'duration_quarter', 'onset_div', 'duration_div', 'pitch', 'voice', 'id', 'staff', 'divs_pq')

First 20 notes:
0 {'onset_beat': 0.0, 'duration_beat': 2.0, 'onset_quarter': 0.0, 'duration_quarter': 4.0, 'onset_div': 0, 'duration_div': 48, 'pitch': 37, 'voice': 5, 'id': 'p0n1', 'staff': 2, 'divs_pq': 12}
1 {'onset_beat': 0.0, 'duration_beat': 2.0, 'onset_quarter': 0.0, 'duration_quarter': 4.0, 'onset_div': 0, 'duration_div': 48, 'pitch': 49, 'voice': 5, 'id': 'p0n2', 'staff': 2, 'divs_pq': 12}
2 {'onset_beat': 0.0, 'duration_beat': 0.1666666716337204, 'onset_quarter': 0.0, 'duration_quarter': 0.3333333432674408, 'onset_div': 0, 'duration_div': 4, 'pitch': 56, 'voice': 1, 'id': 'p0n0', 'staff': 1, 'divs_pq': 12}
3 {'onset_beat': 0.1666666716337204, 'duration_beat': 0.1666666716337204, 'onset_quarter': 0.3333333432674408, 'duration_quarter': 0.3333333432674408, 'onset_div': 4, 'duration_div': 4, 'pitch': 61, 'voice': 1, 'id'

In [3]:
import xml.etree.ElementTree as ET
from pathlib import Path

musicxml_path = Path("dataset/raw/Untitled score.musicxml")

tree = ET.parse(musicxml_path)
root = tree.getroot()

def ns_tag(tag):
    if root.tag.startswith("{"):
        uri = root.tag.split("}")[0][1:]
        return f"{{{uri}}}{tag}"
    return tag

current_time = 0
note_index = 0

for measure in root.findall(f".//{ns_tag('measure')}"):
    measure_no = measure.attrib.get("number", "?")

    for note in measure.findall(ns_tag("note")):
        is_rest = note.find(ns_tag("rest")) is not None
        is_chord_continuation = note.find(ns_tag("chord")) is not None

        if not is_chord_continuation:
            # new onset
            current_time += 1

        pitch_elem = note.find(ns_tag("pitch"))
        if pitch_elem is not None:
            step = pitch_elem.findtext(ns_tag("step"), "")
            alter = pitch_elem.findtext(ns_tag("alter"), "")
            octave = pitch_elem.findtext(ns_tag("octave"), "")
            pitch = f"{step}{alter}{octave}"
        elif is_rest:
            pitch = "REST"
        else:
            pitch = "UNKNOWN"

        lyrics = []
        for lyric in note.findall(ns_tag("lyric")):
            number = lyric.attrib.get("number", "")
            text = lyric.findtext(ns_tag("text"), "")
            lyrics.append((number, text))

        print({
            "note_index": note_index,
            "measure": measure_no,
            "onset_group": current_time,
            "pitch": pitch,
            "is_chord_continuation": is_chord_continuation,
            "lyrics": lyrics,
        })

        note_index += 1

{'note_index': 0, 'measure': '1', 'onset_group': 1, 'pitch': 'E4', 'is_chord_continuation': False, 'lyrics': [('1', 'S')]}
{'note_index': 1, 'measure': '1', 'onset_group': 2, 'pitch': 'REST', 'is_chord_continuation': False, 'lyrics': []}
{'note_index': 2, 'measure': '1', 'onset_group': 3, 'pitch': 'REST', 'is_chord_continuation': False, 'lyrics': []}
{'note_index': 3, 'measure': '1', 'onset_group': 4, 'pitch': 'REST', 'is_chord_continuation': False, 'lyrics': []}
{'note_index': 4, 'measure': '2', 'onset_group': 5, 'pitch': 'C4', 'is_chord_continuation': False, 'lyrics': [('1', 'C')]}
{'note_index': 5, 'measure': '2', 'onset_group': 5, 'pitch': 'E4', 'is_chord_continuation': True, 'lyrics': []}
{'note_index': 6, 'measure': '2', 'onset_group': 5, 'pitch': 'G4', 'is_chord_continuation': True, 'lyrics': []}
{'note_index': 7, 'measure': '2', 'onset_group': 6, 'pitch': 'REST', 'is_chord_continuation': False, 'lyrics': []}
{'note_index': 8, 'measure': '2', 'onset_group': 7, 'pitch': 'REST', '

In [ ]:
from pathlib import Path
import pandas as pd


INPUT_CSV = "dataset/labels/test_labels3.csv"

LABEL_COLS = ["scale", "arpeggio", "chord", "jump"]
REQUIRED_COLS = [
    "part_index",
    "part_id",
    "measure",
    "xml_note_index",
    "pitch_name",
    "pitch_midi",
    "staff",
    "voice",
    "lyric",
    "scale",
    "arpeggio",
    "chord",
    "jump",
]


def main():
    input_path = Path(INPUT_CSV)

    if not input_path.exists():
        raise FileNotFoundError(f"CSV not found: {input_path}")

    df = pd.read_csv(input_path)

    errors = []

    # 1. Check required columns
    for col in REQUIRED_COLS:
        if col not in df.columns:
            errors.append(f"Missing required column: {col}")

    if errors:
        print("Validation failed.")
        for error in errors:
            print("-", error)
        return

    # 2. Check label values are only 0 or 1
    for col in LABEL_COLS:
        bad_rows = df[~df[col].isin([0, 1])]
        if len(bad_rows) > 0:
            errors.append(f"Column {col} has values other than 0 or 1.")

    # 3. Check missing pitch_midi
    if df["pitch_midi"].isna().any():
        errors.append("Some rows have missing pitch_midi.")

    # 4. Check duplicate xml_note_index
    duplicated = df[df["xml_note_index"].duplicated(keep=False)]
    if len(duplicated) > 0:
        errors.append("Duplicate xml_note_index found.")

    if errors:
        print("Validation failed.")
        for error in errors:
            print("-", error)
        return

    print("Validation passed.")
    print()
    print(f"Rows: {len(df)}")

    print()
    print("Label counts:")
    print(df[LABEL_COLS].sum())

    none_count = (df[LABEL_COLS].sum(axis=1) == 0).sum()

    print()
    print(f"None / all-zero rows: {none_count}")

    print()
    print("Lyric values used:")
    print(df["lyric"].value_counts(dropna=False))


if __name__ == "__main__":
    main()

Validation passed.

Rows: 36

Label counts:
scale       8
arpeggio    4
chord       9
jump        2
dtype: int64

None / all-zero rows: 13

Lyric values used:
lyric
N    13
C     9
S     8
A     4
J     2
Name: count, dtype: int64
